# AMI Knowledge Graph Query Notebook

This notebook queries the Neo4j graph built in `01_kg_construction.ipynb`. It covers direct Cypher checks and GraphRAG text-to-Cypher querying for courses, labs, and news mentions.


## Graph Schema

Nodes:
- `Faculty {name, url}`
- `Department {name}`
- `Person {name, title, position, email, profile_slug, profile_url}`
- `Course {course_id, name, url}`
- `Lab {name, url, email, phone, phone_url}`
- `NewsArticle {url, title, published_at, preview, content, source_page}`

Relationships:
- `(:Faculty)-[:HAS_DEPARTMENT]->(:Department)`
- `(:Department)-[:EMPLOYS]->(:Person)`
- `(:Person)-[:TEACHES]->(:Course)`
- `(:Faculty)-[:HAS_LAB]->(:Lab)`
- `(:Person)-[:LEADS]->(:Lab)`
- `(:NewsArticle)-[:MENTIONS]->(:Person)`


In [72]:
%pip install -r requirements.txt


Note: you may need to restart the kernel to use updated packages.


In [73]:
import json
import os
from typing import Any

from dotenv import load_dotenv
from IPython.display import Markdown, display
from neo4j import GraphDatabase
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.retrievers import Text2CypherRetriever

from graphrag_build_kg import require_env
from rate_limit_utils import configure_rate_limit, retry_with_backoff, wrap_openai_compatible_client

load_dotenv()

OPENAI_BASE_URL = os.getenv('OPENAI_BASE_URL')
OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4.1-mini')
DATABASE = os.getenv('NEO4J_DATABASE')
API_SLEEP_SECONDS = float(os.getenv('NVIDIA_API_SLEEP_SECONDS', '1.0'))
API_MAX_RETRIES = int(os.getenv('NVIDIA_API_MAX_RETRIES', '6'))
API_BACKOFF_SECONDS = float(os.getenv('NVIDIA_API_BACKOFF_SECONDS', '4.0'))

configure_rate_limit(
    sleep_seconds=API_SLEEP_SECONDS,
    max_retries=API_MAX_RETRIES,
    backoff_seconds=API_BACKOFF_SECONDS,
)

def get_driver():
    return GraphDatabase.driver(
        require_env('NEO4J_URI'),
        auth=(require_env('NEO4J_USERNAME'), require_env('NEO4J_PASSWORD')),
    )

def run_cypher(query: str, params: dict[str, Any] | None = None) -> list[dict[str, Any]]:
    driver = get_driver()
    try:
        with driver.session(database=DATABASE) as session:
            return [record.data() for record in session.run(query, params or {})]
    finally:
        driver.close()


## Direct Cypher Queries

Use these first to confirm the graph looks healthy.


In [74]:
run_cypher('''
MATCH (n)
RETURN labels(n) AS labels, count(*) AS total
ORDER BY total DESC
''')


[{'labels': ['Course'], 'total': 158},
 {'labels': ['Person'], 'total': 102},
 {'labels': ['Department'], 'total': 8},
 {'labels': ['NewsArticle'], 'total': 8},
 {'labels': ['Lab'], 'total': 6},
 {'labels': ['Faculty'], 'total': 1}]

In [75]:
run_cypher('''
MATCH (d:Department)-[:EMPLOYS]->(p:Person)-[:TEACHES]->(c:Course)
RETURN d.name AS department, p.name AS person, c.name AS course
ORDER BY department, person, course
LIMIT 25
''')


[{'department': 'Department of Applied Mathematics',
  'person': 'Ihor Makar',
  'course': 'Parallel and distributed systems'},
 {'department': 'Department of Applied Mathematics',
  'person': 'Lesia Fedak',
  'course': 'Object Oriented Software Design (am)'},
 {'department': 'Department of Applied Mathematics',
  'person': 'Liliia Diakoniuk',
  'course': 'Algorithms for Computing Processes (Applied Mathematics)'},
 {'department': 'Department of Applied Mathematics',
  'person': 'Liliia Diakoniuk',
  'course': 'Computational linear algebra (cs)'},
 {'department': 'Department of Applied Mathematics',
  'person': 'Liliia Diakoniuk',
  'course': 'Object Oriented Software Design (am)'},
 {'department': 'Department of Applied Mathematics',
  'person': 'Mykhaylo Shcherbatyy',
  'course': 'Applied Statistical Modeling (am)'},
 {'department': 'Department of Applied Mathematics',
  'person': 'Mykhaylo Shcherbatyy',
  'course': 'Computational Methods'},
 {'department': 'Department of Applied Mat

In [76]:
run_cypher('''
MATCH (p:Person)-[:LEADS]->(l:Lab)
RETURN p.name AS person, l.name AS lab, l.email AS email, l.phone AS phone
ORDER BY lab
''')


[{'person': 'L. B. Pakholkiv',
  'lab': 'Educational Laboratory of Computer Modeling',
  'email': None,
  'phone': '(032) 239-43-91'},
 {'person': 'M. P. Onysko',
  'lab': 'Information Systems',
  'email': None,
  'phone': None},
 {'person': 'V. B. Pasichnyk',
  'lab': 'Information Technology',
  'email': 'itl@lnu.edu.ua',
  'phone': '(032) 239-44-27'},
 {'person': 'L. B. Halamaha',
  'lab': 'Programming',
  'email': None,
  'phone': None},
 {'person': 'O. B. Tomchii',
  'lab': 'System Analysis',
  'email': None,
  'phone': '(032) 239-46-09'}]

In [77]:
run_cypher('''
MATCH (n:NewsArticle)-[:MENTIONS]->(p:Person)
RETURN p.name AS person, n.title AS article, n.published_at AS published_at, n.url AS url
ORDER BY published_at DESC, article
LIMIT 25
''')


[{'person': 'Ihor Makar',
  'article': 'We invite you to an open lecture by Assoc. Prof. Igor Makar',
  'published_at': '24.04.2025 | 05:32',
  'url': 'https://ami.lnu.edu.ua/en/news/we-invite-you-to-an-open-lecture-by-assoc-prof-igor-makar'},
 {'person': 'Ihor Makar',
  'article': 'Open Lecture by Associate Professor of the Department of Applied Mathematics Ihor Makar',
  'published_at': '18.03.2026 | 09:44',
  'url': 'https://ami.lnu.edu.ua/en/news/open-lecture-by-associate-professor-of-the-department-of-applied-mathematics-ihor-makar'}]

## GraphRAG Text-To-Cypher Setup


In [78]:
GRAPH_SCHEMA = '''
Node properties:
Faculty {name, url}
Department {name}
Person {name, title, position, email, profile_slug, profile_url}
Course {course_id, name, url}
Lab {name, url, email, phone, phone_url}
NewsArticle {url, title, published_at, preview, content, source_page}

Relationship properties:
HAS_DEPARTMENT {}
EMPLOYS {}
TEACHES {}
HAS_LAB {}
LEADS {}
MENTIONS {}

The relationships are:
(:Faculty)-[:HAS_DEPARTMENT]->(:Department)
(:Department)-[:EMPLOYS]->(:Person)
(:Person)-[:TEACHES]->(:Course)
(:Faculty)-[:HAS_LAB]->(:Lab)
(:Person)-[:LEADS]->(:Lab)
(:NewsArticle)-[:MENTIONS]->(:Person)
'''

EXAMPLES = [
    "USER INPUT: Which labs belong to the faculty? QUERY: MATCH (:Faculty)-[:HAS_LAB]->(l:Lab) RETURN l.name AS lab, l.email AS email, l.phone AS phone ORDER BY l.name",
    "USER INPUT: Who leads the Information Technology lab? QUERY: MATCH (p:Person)-[:LEADS]->(l:Lab {name: 'Information Technology'}) RETURN p.name AS person, p.profile_url AS profile_url",
    "USER INPUT: Which courses does Mariia Smychok teach? QUERY: MATCH (p:Person {name: 'Mariia Smychok'})-[:TEACHES]->(c:Course) RETURN c.name AS course, c.url AS url ORDER BY c.name",
    "USER INPUT: Which news articles mention Vasyl Pasichnyk? QUERY: MATCH (n:NewsArticle)-[:MENTIONS]->(p:Person {name: 'Vasyl Pasichnyk'}) RETURN n.title AS article, n.published_at AS published_at, n.url AS url ORDER BY published_at DESC, article",
    "USER INPUT: Which people are mentioned in news articles? QUERY: MATCH (n:NewsArticle)-[:MENTIONS]->(p:Person) RETURN DISTINCT p.name AS person ORDER BY person",
]

CUSTOM_PROMPT = '''
Task: Generate a Cypher query for the provided graph schema.

Schema:
{schema}

Examples:
{examples}

User question:
{query_text}

Rules:
- Return only valid Cypher.
- Use only the labels and relationship types from the schema.
- Prefer exact string matching on names from the graph.
'''


In [79]:
llm = OpenAILLM(
    model_name=OPENAI_MODEL,
    model_params={'temperature': 0},
    base_url=OPENAI_BASE_URL,
)
wrap_openai_compatible_client(getattr(llm, 'client', None))

driver = get_driver()
retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=GRAPH_SCHEMA,
    examples=EXAMPLES,
    custom_prompt=CUSTOM_PROMPT,
    neo4j_database=DATABASE,
)
rag = GraphRAG(retriever=retriever, llm=llm)


In [85]:
def execute_cypher(cypher: str) -> list[dict[str, Any]]:
    with driver.session(database=DATABASE) as session:
        return [record.data() for record in session.run(cypher)]

def make_jsonable(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, dict):
        return {str(key): make_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [make_jsonable(item) for item in value]
    if hasattr(value, 'model_dump'):
        return make_jsonable(value.model_dump())
    if hasattr(value, 'dict'):
        return make_jsonable(value.dict())
    if hasattr(value, '__dict__'):
        return make_jsonable(vars(value))
    return repr(value)

def extract_llm_context(response: Any, records: list[dict[str, Any]]) -> dict[str, Any]:
    retriever_result = getattr(response, 'retriever_result', None)
    metadata = getattr(retriever_result, 'metadata', {}) or {}
    context = {
        'response_context': make_jsonable(getattr(response, 'context', None)),
        'retriever_items': make_jsonable(getattr(retriever_result, 'items', None)),
        'retriever_records': make_jsonable(getattr(retriever_result, 'records', None)),
        'records_used_for_answer': make_jsonable(records),
        'retriever_metadata': make_jsonable(metadata),
    }
    return {key: value for key, value in context.items() if value not in (None, [], {}, '')}

def format_answer_from_records(question: str, records: list[dict[str, Any]]) -> str:
    if not records:
        return 'No matching information found.'

    prompt = f'''You are a data assistant.

User question:
{question}

Database query result (JSON):
{json.dumps(records, ensure_ascii=False, indent=2)}

Instructions:
- Answer the question using ONLY the provided data.
- Be concise and accurate.
- If the data contains the answer, state it directly.
- If the user asked for a list, format it cleanly.
- Do not say the context is missing if the data is present.
- Do not hallucinate.
'''

    return llm.invoke(prompt).content.strip()

def ask_with_graphrag(question: str, show_context: bool = False, show_raw_answer: bool = False) -> dict[str, Any]:
    response = retry_with_backoff(
        rag.search,
        query_text=question,
        return_context=True,
    )
    metadata = getattr(response.retriever_result, 'metadata', {})
    cypher = metadata.get('cypher')
    records = execute_cypher(cypher) if cypher else []
    llm_context = extract_llm_context(response, records)

    formatted_answer = format_answer_from_records(question, records)
    display(Markdown(f'''## Answer

{formatted_answer}

```'''))

    if show_context:
        print('LLM context:')
        print(json.dumps(llm_context, ensure_ascii=False, indent=2))

    result = {
        'question': question,
        'answer': formatted_answer,
        'cypher': cypher,
        'records': records,
        'llm_context': llm_context,
    }
    if show_raw_answer:
        result['raw_graphrag_answer'] = response.answer
    #return result


## Example GraphRAG Questions


In [86]:
ask_with_graphrag('Which labs belong to the faculty?')


## Answer

The labs that belong to the faculty are:

- Educational Laboratory of Computer Modeling
- Information Systems
- Information Technology
- Mathematical and Computer Modeling
- Programming
- System Analysis

```

In [87]:
ask_with_graphrag('Which courses Yurii Muzychuk have?')

## Answer

Yurii Muzychuk has the following courses:

1. Boundary element method  
   [Course link](https://ami.lnu.edu.ua/en/course/boundary-element-method)

2. Deep Learning Models (Applied Mathematics, 1.9)  
   [Course link](https://ami.lnu.edu.ua/en/course/deep-learning-models-applied-mathematics-1-9)

3. Deep learning models (am)  
   [Course link](https://ami.lnu.edu.ua/en/course/deep-learning-models)

4. Fundamentals of machine learning (AM)  
   [Course link](https://ami.lnu.edu.ua/en/course/fundamentals-of-machine-learning-am)

```

In [88]:
ask_with_graphrag('Which courses Yuriy Yashchuk have?')

## Answer

Yuriy Yashchuk has the following courses:

1. Numerical Methods of Mathematical Physics (am)  
   [Course link](https://ami.lnu.edu.ua/en/course/numerical-methods-of-mathematical-physics-applied-mathematics)

2. Software (am)  
   [Course link](https://ami.lnu.edu.ua/en/course/software-applied-mathematics)

```

In [89]:
ask_with_graphrag('Who leads the Information Technology lab?')


## Answer

The Information Technology lab is led by V. B. Pasichnyk.

```

In [90]:
ask_with_graphrag('Return the list of top 5 teachers with the most courses')

## Answer

Top 5 teachers with the most courses:

1. Mykhaylo Shcherbatyy - 10 courses  
2. Serhiy Yaroshko - 9 courses  
3. Anastasiia Nedashkovska - 8 courses  
4. Valeriy Trushevskyy - 8 courses  
5. Yaroslav Harasym - 8 courses

```

In [91]:
ask_with_graphrag('Which departments belong to the faculty?')

## Answer

The departments belonging to the faculty are:

- Department of Applied Mathematics  
- Department of Computational Mathematics  
- Department of Cybersecurity  
- Department of Discrete Analysis and Intelligent System  
- Department of Information Systems  
- Department of Mathematical Modeling of Social and Economics Processes  
- Department of Programming

```

In [92]:
ask_with_graphrag('Which laboratories or research units exist at the faculty?')

## Answer

The faculty has the following laboratories or research units:

- Educational Laboratory of Computer Modeling, phone: (032) 239-43-91
- Information Systems
- Information Technology, email: itl@lnu.edu.ua, phone: (032) 239-44-27
- Mathematical and Computer Modeling, phone: (032) 239-45-09
- Programming
- System Analysis, phone: (032) 239-46-09

```

In [93]:
ask_with_graphrag('What are the latest news?')

## Answer

The latest news are:

1. Open lecture by Associate Professor Lesia Petrivna Dobuliak (31.03.2026)  
   [Read more](https://ami.lnu.edu.ua/en/news/open-lecture-by-associate-professor-lesia-petrivna-dobuliak)

2. Open Lecture by Associate Professor of the Department of Applied Mathematics, Andrii Pereimybida (24.03.2026)  
   [Read more](https://ami.lnu.edu.ua/en/news/open-lecture-by-associate-professor-of-the-department-of-applied-mathematics-andrii-pereimybida)

3. Open Lecture by Associate Professor of the Department of Applied Mathematics Ihor Makar (18.03.2026)  
   [Read more](https://ami.lnu.edu.ua/en/news/open-lecture-by-associate-professor-of-the-department-of-applied-mathematics-ihor-makar)

```

In [94]:
driver.close()